# 3D Latent Diffusion Model
In this tutorial, we will walk through the process of using the MONAI Generative Models package to generate synthetic data using Latent Diffusion Models (LDM)  [1, 2]. Specifically, we will focus on training an LDM to create synthetic brain images from the Brats dataset.

[1] - Rombach et al. "High-Resolution Image Synthesis with Latent Diffusion Models" https://arxiv.org/abs/2112.10752

[2] - Pinaya et al. "Brain imaging generation with latent diffusion models" https://arxiv.org/abs/2209.07162

### Set up imports

In [ ]:
!nvidia-smi
!jupyter notebook --NotebookApp.iopub_data_rate_limit=1000000000

In [ ]:
!pip install matplotlib scikit-learn
!pip install nibabel pandas transformers einops
!pip install monai
!pip install monai-generative #Install 'monai-generative' instead of 'monai generative'
!pip install lpips
!pip install numpy

In [ ]:
!pip install --upgrade numpy

In [ ]:

import os
import shutil
import tempfile

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from monai import transforms
from monai.apps import DecathlonDataset
from monai.config import print_config
from monai.data import DataLoader
from monai.utils import first, set_determinism
from torch.cuda.amp import GradScaler, autocast
from torch.nn import L1Loss
from tqdm import tqdm

from generative.inferers import LatentDiffusionInferer
from generative.losses import PatchAdversarialLoss, PerceptualLoss
from generative.networks.nets import AutoencoderKL, DiffusionModelUNet, PatchDiscriminator
from generative.networks.schedulers import DDPMScheduler

print_config()

In [ ]:
# for reproducibility purposes set a seed
set_determinism(42)

### Setup a data directory and download dataset
Specify a MONAI_DATA_DIRECTORY variable, where the data will be downloaded. If not specified a temporary directory will be used.

In [ ]:
directory = "data"
root_dir = tempfile.mkdtemp() if directory is None else directory
print(root_dir)

In [ ]:
import pandas as pd
import os
import numpy as np
from sklearn.preprocessing import OneHotEncoder

# Path to TSV file and image directory
tsv_path = "participants_custom.tsv"  # Update this path
image_dir = "data"                  # Update this path

# Define fixed categories for one-hot encoding
fixed_categories = {
    "wab_type": ["Anomic", "Broca", "Conduction", "Global", "TranscorticalMotor", "Wernicke", "None", "Unknown"],
    "sex": ["F", "M"],
    "race": ["b", "w"]  # Extend this if there are other race categories
}

# Load data
data = pd.read_csv(tsv_path, sep='\t')

# Replace missing values in 'wab_type' and 'race' with "Unknown"
data['wab_type'] = data['wab_type'].fillna("Unknown")
data['race'] = data['race'].fillna("Unknown")

# Normalize numerical variables by dividing by 100
data['wab_aq_normalized'] = data['wab_aq'] / 100.0
data['age_at_stroke_normalized'] = data['age_at_stroke'] / 100.0

# Initialize OneHotEncoders with fixed categories and handle unknown categories
wab_type_encoder = OneHotEncoder(categories=[fixed_categories["wab_type"]], sparse_output=False, handle_unknown="ignore")
sex_encoder = OneHotEncoder(categories=[fixed_categories["sex"]], sparse_output=False, handle_unknown="ignore")
race_encoder = OneHotEncoder(categories=[fixed_categories["race"]], sparse_output=False, handle_unknown="ignore")

# Encode categorical variables
wab_type_encoded = wab_type_encoder.fit_transform(data[['wab_type']])
sex_encoded = sex_encoder.fit_transform(data[['sex']])
race_encoded = race_encoder.fit_transform(data[['race']])


# Combine all condition vectors
conditions = np.hstack((
    wab_type_encoded,  # One-hot encoded WAB type
    sex_encoded,       # One-hot encoded sex
    race_encoded,      # One-hot encoded race
    data[['wab_aq_normalized', 'age_at_stroke_normalized']].values  # Normalized numerical variables
))

# Check alignment between DataFrame and conditions array
if len(data) != len(conditions):
    raise ValueError("Mismatch between data rows and condition vectors!")

# Function to match participant IDs to files
def match_participant_to_files(participant_id, image_dir):
    """
    Matches the participant ID to all files in the directory based on file naming convention.
    """
    matching_files = []
    for filename in os.listdir(image_dir):
        if participant_id in filename:  # Match based on participant ID in the filename
            matching_files.append(os.path.join(image_dir, filename))
    return matching_files

# Create formatted data with condition vectors and image paths
formatted_data = []
for idx, row in data.iterrows():
    participant_id = row['participant_id']
    matched_files = match_participant_to_files(participant_id, image_dir)
    if matched_files:  # Only include participants with matching files
        formatted_data.append({
            "images": matched_files,
            "condition": conditions[idx].tolist()
        })

# Display example formatted data
print("\nFormatted Dataset Example:")
for entry in formatted_data[:5]:  # Show first 5 entries
    print(entry)

# Save formatted data as JSON
formatted_data_path = "formatted_data.json"  # Update this path
pd.DataFrame(formatted_data).to_json(formatted_data_path, orient="records", indent=2)
print(f"\nFormatted data saved to {formatted_data_path}")


In [ ]:
import os
import json

# Define paths
dataset_path = "data"
formatted_data_path = "formatted_data.json"  # Path to the existing formatted JSON

# Load the existing formatted data (from JSON file)
with open(formatted_data_path, 'r') as f:
    formatted_data = json.load(f)

# Get the list of subject directories starting with "sub-"
subject_dirs = [d for d in os.listdir(dataset_path) if d.startswith("sub-")]

# Sort the directories to ensure a consistent order
subject_dirs.sort()

# Rename each subject directory and update the JSON file paths
for i, subject in enumerate(subject_dirs, start=1):
    old_path = os.path.join(dataset_path, subject)
    new_name = f"sub-{i}.nii.gz"
    new_path = os.path.join(dataset_path, new_name)

    # Rename the directory
    os.rename(old_path, new_path)
    print(f"Renamed {old_path} to {new_path}")

    # Update the corresponding file paths in the formatted_data
    for entry in formatted_data:
        entry["images"] = [new_path if subject in file else file for file in entry["images"]]

# Save the updated formatted data back into the JSON file
with open(formatted_data_path, 'w') as f:
    json.dump(formatted_data, f, indent=2)

print("Renaming completed and JSON file updated!")


### Prepare data loader for the training set
Here we will download the Brats dataset using MONAI's `DecathlonDataset` class, and we prepare the data loader for the training set.

In [ ]:
from monai.data import Dataset, DataLoader
from monai.transforms import (
    LoadImaged,
    EnsureChannelFirstd,
    Spacingd,
    Orientationd,
    ScaleIntensityRangePercentilesd,
    CenterSpatialCropd,
    EnsureTyped,
)

# Define your data directory
data_dir = "/content/data"

# Prepare a list of data dictionaries
# Assuming filenames are of the format sub-{i}.nii.gz
data_list = [{"image": f"{data_dir}/sub-{i}.nii.gz"} for i in range(30)]

# Define the transformations
train_transforms = transforms.Compose(
    [
        LoadImaged(keys=["image"]),
        EnsureChannelFirstd(keys=["image"]),
        Orientationd(keys=["image"], axcodes="RAS"),
        Spacingd(keys=["image"], pixdim=(2.4, 2.4, 2.2), mode=("bilinear")),
        CenterSpatialCropd(keys=["image"], roi_size=(107, 107, 96)),
        ScaleIntensityRangePercentilesd(
            keys=["image"], lower=0, upper=99.5, b_min=0, b_max=1
        ),
        EnsureTyped(keys=["image"]),
    ]
)

# Create a custom dataset and data loader
train_ds = Dataset(data=data_list, transform=train_transforms)
train_loader = DataLoader(train_ds, batch_size=10, shuffle=True, num_workers=4)

print(f"Number of samples: {len(train_ds)}")


### Visualise examples from the training set

In [ ]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from monai import transforms
from monai.networks.nets import AutoencoderKL
from tqdm import tqdm

# Define constants
DATA_DIR = "data"
IMAGE_SIZE = (96, 96, 96)
BATCH_SIZE = 10
NUM_WORKERS = 10
LEARNING_RATE = 1e-2
NUM_EPOCHS = 30

# Custom dataset
class BrainMRIDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir
        self.transform = transform
        self.file_list = [f for f in os.listdir(data_dir) if f.endswith('.nii.gz')]

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        img_path = os.path.join(self.data_dir, self.file_list[idx])
        image = {'image': img_path}
        if self.transform:
            image = self.transform(image)
        return image

# Define transformations
train_transforms = transforms.Compose([
    transforms.LoadImaged(keys=["image"]),
    transforms.EnsureChannelFirstd(keys=["image"]),
    transforms.Orientationd(keys=["image"], axcodes="RAS"),
    transforms.Spacingd(keys=["image"], pixdim=(2.4, 2.4, 2.2), mode=("bilinear")),
    transforms.ResizeWithPadOrCropd(keys=["image"], spatial_size=IMAGE_SIZE),
    transforms.ScaleIntensityRangePercentilesd(keys=["image"], lower=0, upper=99.5, b_min=0, b_max=1),
    transforms.EnsureTyped(keys=["image"]),
])

# Create dataset and dataloader
train_ds = BrainMRIDataset(DATA_DIR, transform=train_transforms)
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=8)


In [ ]:
check_data = first(train_loader)
print(check_data["image"].shape)  # Should be consistent for all samples

In [ ]:
idx = 1
img = check_data["image"][idx, 0]  # Accessing the first image

fig, axs = plt.subplots(nrows=1, ncols=3)
axs[0].imshow(img[..., img.shape[2] // 2], cmap="gray")  # Axial slice
axs[1].imshow(img[:, img.shape[1] // 2, ...], cmap="gray")  # Coronal slice
axs[2].imshow(img[img.shape[0] // 2, ...], cmap="gray")  # Sagittal slice
plt.show()


## Autoencoder KL

### Define Autoencoder KL network

In this section, we will define an autoencoder with KL-regularization for the LDM. The autoencoder's primary purpose is to transform input images into a latent representation that the diffusion model will subsequently learn. By doing so, we can decrease the computational resources required to train the diffusion component, making this approach suitable for learning high-resolution medical images.


In [ ]:
import torch

device = torch.device("cuda")
print(f"Using {device}")
print(f"Memory allocated: {torch.cuda.memory_allocated(device=0)} bytes")
print(f"Memory reserved: {torch.cuda.memory_reserved(device=0)} bytes")
print("Tony here")


In [ ]:
from einops.layers.torch import Rearrange

import torch
import torch.nn as nn

autoencoder = AutoencoderKL(
    spatial_dims=3,
    in_channels=1,
    out_channels=1,
    channels=(96, 96, 96),
    latent_channels=3,  # Make sure this is 3
    num_res_blocks=1,
    norm_num_groups=4,
    attention_levels=(False, False, False),
)
autoencoder.to(device)

discriminator = PatchDiscriminator(
    spatial_dims=3,
    num_layers_d=3,
    num_channels=32,
    in_channels=1,
    out_channels=1,
)
discriminator.to(device)


## Verify Directory



### Defining Losses

We will also specify the perceptual and adversarial losses, including the involved networks, and the optimizers to use during the training process.

In [ ]:
l1_loss = L1Loss()
adv_loss = PatchAdversarialLoss(criterion="least_squares")
loss_perceptual = PerceptualLoss(spatial_dims=3, network_type="squeeze", is_fake_3d=True, fake_3d_ratio=0.2)
loss_perceptual.to(device)


def KL_loss(z_mu, z_sigma):
    kl_loss = 0.5 * torch.sum(z_mu.pow(2) + z_sigma.pow(2) - torch.log(z_sigma.pow(2)) - 1, dim=[1, 2, 3, 4])
    return torch.sum(kl_loss) / kl_loss.shape[0]


adv_weight = 0.01
perceptual_weight = 0.001
kl_weight = 1e-10

In [ ]:
optimizer_g = torch.optim.Adam(params=autoencoder.parameters(), lr=1e-4)
optimizer_d = torch.optim.Adam(params=discriminator.parameters(), lr=1e-4)

### Train model

In [ ]:
# Full training and visualization code in one cell
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm

# Initialize variables
n_epochs = 300
autoencoder_warm_up_n_epochs = 10
val_interval = 10
epoch_recon_loss_list = []
epoch_gen_loss_list = []
epoch_disc_loss_list = []
n_example_images = 10
img = None  # Placeholder for the reconstruction image

# Start training loop
for epoch in range(n_epochs):
    autoencoder.train()
    discriminator.train()
    epoch_loss = 0
    gen_epoch_loss = 0
    disc_epoch_loss = 0.04
    progress_bar = tqdm(enumerate(train_loader), total=len(train_loader), ncols=110)
    progress_bar.set_description(f"Epoch {epoch}")

    for step, batch in progress_bar:
        images = batch["image"].to(device)  # choose only one of Brats channels

        # Generator part
        optimizer_g.zero_grad()
        reconstruction, z_mu, z_sigma = autoencoder(images)
        kl_loss = KL_loss(z_mu, z_sigma)

        recons_loss = l1_loss(reconstruction.float(), images.float())
        p_loss = loss_perceptual(reconstruction.float(), images.float())
        loss_g = recons_loss + kl_weight * kl_loss + perceptual_weight * p_loss

        if epoch > autoencoder_warm_up_n_epochs:
            logits_fake = discriminator(reconstruction.contiguous().float())[-1]
            generator_loss = adv_loss(logits_fake, target_is_real=True, for_discriminator=False)
            loss_g += adv_weight * generator_loss

        loss_g.backward()
        optimizer_g.step()

        if epoch > autoencoder_warm_up_n_epochs:
            # Discriminator part
            optimizer_d.zero_grad()
            logits_fake = discriminator(reconstruction.contiguous().detach())[-1]
            loss_d_fake = adv_loss(logits_fake, target_is_real=False, for_discriminator=True)
            logits_real = discriminator(images.contiguous().detach())[-1]
            loss_d_real = adv_loss(logits_real, target_is_real=True, for_discriminator=True)
            discriminator_loss = (loss_d_fake + loss_d_real) * 0.5

            loss_d = adv_weight * discriminator_loss

            loss_d.backward()
            optimizer_d.step()

        epoch_loss += recons_loss.item()
        if epoch > autoencoder_warm_up_n_epochs:
            gen_epoch_loss += generator_loss.item()
            disc_epoch_loss += discriminator_loss.item()

        progress_bar.set_postfix(
            {
                "recons_loss": epoch_loss / (step + 1),
                "gen_loss": gen_epoch_loss / (step + 1),
                "disc_loss": disc_epoch_loss / (step + 1),
            }
        )

        # Store one image of the reconstruction from the final batch of the last epoch
        if epoch == n_epochs - 1 and step == len(train_loader) - 1:
            img = reconstruction[0, 0].detach().cpu().numpy()  # Storing a sample (idx=0, channel=0)

        # Delete intermediate tensors to save memory
        del reconstruction, z_mu, z_sigma, recons_loss, p_loss, loss_g

        # Delete logits_fake and generator_loss only if they were defined
        if epoch > autoencoder_warm_up_n_epochs:
            del logits_fake, generator_loss

    epoch_recon_loss_list.append(epoch_loss / (step + 1))
    epoch_gen_loss_list.append(gen_epoch_loss / (step + 1))
    epoch_disc_loss_list.append(disc_epoch_loss / (step + 1))

    # Clear memory after each epoch
    torch.cuda.empty_cache()

# Optionally delete models or other large objects after training
del discriminator, autoencoder, optimizer_g, optimizer_d

# Now visualize the image after training is done
if img is not None:
    # Create subplots for the 3 views: axial, coronal, and sagittal
    fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(15, 5))

    # Plot axial slice (slice along the Z-axis)
    axs[0].imshow(img[..., img.shape[2] // 2], cmap="gray")  # Axial (middle slice in depth)
    axs[0].axis("off")
    axs[0].set_title("Axial Slice")

    # Plot coronal slice (slice along the Y-axis)
    axs[1].imshow(img[:, img.shape[1] // 2, ...], cmap="gray")  # Coronal (middle slice in height)
    axs[1].axis("off")
    axs[1].set_title("Coronal Slice")

    # Plot sagittal slice (slice along the X-axis)
    axs[2].imshow(img[img.shape[0] // 2, ...], cmap="gray")  # Sagittal (middle slice in width)
    axs[2].axis("off")
    axs[2].set_title("Sagittal Slice")

    # Show the plot
    plt.suptitle(f'Slices for Image {0}', fontsize=16)
    plt.show()

In [ ]:
plt.style.use("ggplot")
plt.title("Learning Curves", fontsize=20)
plt.plot(epoch_recon_loss_list)
plt.yticks(fontsize=12)
plt.xticks(fontsize=12)
plt.xlabel("Epochs", fontsize=16)
plt.ylabel("Loss", fontsize=16)
plt.legend(prop={"size": 14})
plt.show()

In [ ]:
plt.title("Adversarial Training Curves", fontsize=20)
plt.plot(epoch_gen_loss_list, color="C0", linewidth=2.0, label="Generator")
plt.plot(epoch_disc_loss_list, color="C1", linewidth=2.0, label="Discriminator")
plt.yticks(fontsize=12)
plt.xticks(fontsize=12)
plt.xlabel("Epochs", fontsize=16)
plt.ylabel("Loss", fontsize=16)
plt.legend(prop={"size": 14})
plt.show()

### Visualise reconstructions

In [ ]:
# Plot axial, coronal, and sagittal slices of a training sample at the end of training
idx = 0  # Index of the image you want to visualize
channel = 0  # Channel of the image to visualize (e.g., MRI modality channel)

# Assuming the reconstruction is a 3D image (height, width, depth) after the autoencoder
img = reconstruction[idx, channel].detach().cpu().numpy()

# Create subplots for the 3 views: axial, coronal, and sagittal
fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(15, 5))

# Plot axial slice (slice along the Z-axis)
axs[0].imshow(img[..., img.shape[2] // 2], cmap="gray")  # Axial (middle slice in depth)
axs[0].axis("off")
axs[0].set_title("Axial Slice")

# Plot coronal slice (slice along the Y-axis)
axs[1].imshow(img[:, img.shape[1] // 2, ...], cmap="gray")  # Coronal (middle slice in height)
axs[1].axis("off")
axs[1].set_title("Coronal Slice")

# Plot sagittal slice (slice along the X-axis)
axs[2].imshow(img[img.shape[0] // 2, ...], cmap="gray")  # Sagittal (middle slice in width)
axs[2].axis("off")
axs[2].set_title("Sagittal Slice")

# Show the plot
plt.suptitle(f'Slices for Image {idx}', fontsize=16)
plt.show()

## Diffusion Model

### Define diffusion model and scheduler

In this section, we will define the diffusion model that will learn data distribution of the latent representation of the autoencoder. Together with the diffusion model, we define a beta scheduler responsible for defining the amount of noise that is added across the diffusion's model Markov chain.

In [ ]:
# Calculate the total size of your condition vector
condition_size = 8 + 2 + 2 + 2  # wab_type (8) + sex (2) + race (2) + wab_aq_age (2)

unet = DiffusionModelUNet(
    spatial_dims=3,
    in_channels=3,
    out_channels=3,
    num_res_blocks=1,
    num_channels=(32, 64, 64),
    attention_levels=(False, True, True),
    num_head_channels=(0, 64, 64),
    with_conditioning=True,
    cross_attention_dim=condition_size,
    num_class_embeds=None
)
unet.to(device)

scheduler = DDPMScheduler(num_train_timesteps=1000, schedule="scaled_linear_beta", beta_start=0.0015, beta_end=0.0195)

### Scaling factor

As mentioned in Rombach et al. [1] Section 4.3.2 and D.1, the signal-to-noise ratio (induced by the scale of the latent space) can affect the results obtained with the LDM, if the standard deviation of the latent space distribution drifts too much from that of a Gaussian. For this reason, it is best practice to use a scaling factor to adapt this standard deviation.

_Note: In case where the latent space is close to a Gaussian distribution, the scaling factor will be close to one, and the results will not differ from those obtained when it is not used._


In [ ]:
with torch.no_grad():
    with autocast(enabled=True):
        z = autoencoder.encode_stage_2_inputs(check_data["image"].to(device))
print(f"Scaling factor set to {1/torch.std(z)}")
scale_factor = 1 / torch.std(z)

We define the inferer using the scale factor:

In [ ]:
inferer = LatentDiffusionInferer(scheduler, scale_factor=scale_factor)

In [ ]:
optimizer_diff = torch.optim.Adam(params=unet.parameters(), lr=1e-4)

### Train diffusion model

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import torch.nn.functional as F
import nibabel as nib
import numpy as np
import json
from tqdm import tqdm
import torch.nn as nn

# Dataset class for Brain MRI data
class BrainMRIDataset(Dataset):
    def __init__(self, json_file, transform=None):
        with open(json_file, 'r') as f:
            self.data = json.load(f)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        image_path = item['images'][0]

        # Load NIfTI image
        image_nii = nib.load(image_path)
        image_data = image_nii.get_fdata()  # Extract image data as NumPy array

        # Normalize and convert to torch tensor
        image_data = torch.tensor(image_data, dtype=torch.float32)
        image_data = (image_data - image_data.min()) / (image_data.max() - image_data.min())

        # Add channel dimension for 3D data (C x H x W x D)
        if image_data.ndim == 3:
            image_data = image_data.unsqueeze(0)

        # Apply any transformations
        if self.transform:
            image_data = self.transform(image_data)

        # Process condition vectors
        condition = torch.tensor(item['condition'], dtype=torch.float32)

        return {'image': image_data, 'conditions': condition}

# Custom collate function for batching
def custom_collate(batch):
    images = torch.stack([item['image'] for item in batch])
    conditions = torch.stack([item['conditions'] for item in batch])
    return {'image': images, 'conditions': conditions}

# Transformation class for resizing 3D images
class Resize3D:
    """Resize 3D images to a target shape."""
    def __init__(self, target_shape):
        self.target_shape = target_shape

    def __call__(self, image):
        image = image.unsqueeze(0)  # Add batch dimension
        resized = F.interpolate(image, size=self.target_shape, mode='trilinear', align_corners=False)
        return resized.squeeze(0)  # Remove batch dimension

# Parameters
target_shape = (36, 36, 36)  # Resize target
batch_size = 10
n_epochs = 30
expected_condition_dim = 128  # Define the expected condition dimension (adjust as needed)

# Transformations
train_transforms = Resize3D(target_shape)

# Dataset and DataLoader
train_ds = BrainMRIDataset(json_file="formatted_data.json", transform=train_transforms)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4, collate_fn=custom_collate)



# Gradient scaler for mixed-precision training
scaler = GradScaler()

# Training loop
epoch_loss_list = []

for epoch in range(n_epochs):
    unet.train()
    epoch_loss = 0
    progress_bar = tqdm(enumerate(train_loader), total=len(train_loader), ncols=70)
    progress_bar.set_description(f"Epoch {epoch+1}/{n_epochs}")

    for step, batch in progress_bar:
        images = batch["image"].to(device)
        conditions = batch["conditions"].to(device)

        optimizer_diff.zero_grad(set_to_none=True)

        with autocast(enabled=True):
            # Generate latent representations using autoencoder
            z = autoencoder.encode_stage_2_inputs(images)  # Latent representation
            noise = torch.randn_like(z).to(device)  # Random noise

            # Generate timesteps
            timesteps = torch.randint(
                0, scheduler.num_train_timesteps, (images.shape[0],), device=device
            ).long()

            # Run inference through the diffusion model
            z = z.unsqueeze(1) if z.ndim == 4 else z  # Ensure latent has channel dimension
            noisy_latents = scheduler.add_noise(original_samples=z, noise=noise, timesteps=timesteps)

            # Check tensor shapes
            print(f"noisy_latents shape: {noisy_latents.shape}")  # Should be 5D

            # Reshape conditions to match model input
            batch_size = conditions.shape[0]
            flattened_conditions = conditions.reshape(batch_size, -1)  # Flatten to [batch_size, 14 * 9 * 9 * 9]

            # Apply linear transformation to match model's expected input
            linear_layer = nn.Linear(flattened_conditions.size(1), expected_condition_dim)
            conditions_transformed = linear_layer(flattened_conditions)  # Transformed conditions

            # Pass through the model with the transformed conditions
            noise_pred = unet(x=noisy_latents, timesteps=timesteps, context=conditions_transformed)

            # Compute loss
            loss = F.mse_loss(noise_pred.float(), noise.float())

        # Gradient scaling and optimization
        scaler.scale(loss).backward()
        scaler.step(optimizer_diff)
        scaler.update()

        # Update epoch loss
        epoch_loss += loss.item()
        progress_bar.set_postfix({"loss": epoch_loss / (step + 1)})

    # Append epoch loss
    epoch_loss_list.append(epoch_loss / len(train_loader))
    print(f"Epoch {epoch+1} completed with avg loss: {epoch_loss / len(train_loader):.4f}")

print("Training completed!")


In [ ]:
n_epochs = 100
epoch_loss_list = []
autoencoder.eval()
scaler = GradScaler()

first_batch = first(train_loader)
z = autoencoder.encode_stage_2_inputs(first_batch["image"].to(device))

for epoch in range(n_epochs):
    unet.train()
    epoch_loss = 0
    progress_bar = tqdm(enumerate(train_loader), total=len(train_loader), ncols=70)
    progress_bar.set_description(f"Epoch {epoch}")
    for step, batch in progress_bar:
        images = batch["image"].to(device)
        optimizer_diff.zero_grad(set_to_none=True)

        with autocast(enabled=True):
            # Generate random noise
            noise = torch.randn_like(z).to(device)

            # Create timesteps
            timesteps = torch.randint(
                0, inferer.scheduler.num_train_timesteps, (images.shape[0],), device=images.device
            ).long()

            # Get model prediction
            noise_pred = inferer(
                inputs=images, autoencoder_model=autoencoder, diffusion_model=unet, noise=noise, timesteps=timesteps
            )

            loss = F.mse_loss(noise_pred.float(), noise.float())

        scaler.scale(loss).backward()
        scaler.step(optimizer_diff)
        scaler.update()

        epoch_loss += loss.item()

        progress_bar.set_postfix({"loss": epoch_loss / (step + 1)})
    epoch_loss_list.append(epoch_loss / (step + 1))

In [ ]:
plt.plot(epoch_loss_list)
plt.title("Learning Curves", fontsize=20)
plt.plot(epoch_loss_list)
plt.yticks(fontsize=12)
plt.xticks(fontsize=12)
plt.xlabel("Epochs", fontsize=16)
plt.ylabel("Loss", fontsize=16)
plt.legend(prop={"size": 14})
plt.show()

### Plotting sampling example

Finally, we generate an image with our LDM. For that, we will initialize a latent representation with just noise. Then, we will use the `unet` to perform 1000 denoising steps. In the last step, we decode the latent representation and plot the sampled image.

In [ ]:
autoencoder.eval()
unet.eval()

noise = torch.randn((1, 2, 24, 24, 16))
noise = noise.to(device)
scheduler.set_timesteps(num_inference_steps=1000)
synthetic_images = inferer.sample(
    input_noise=noise, autoencoder_model=autoencoder, diffusion_model=unet, scheduler=scheduler
)

### Visualise synthetic data

In [ ]:
idx = 0
img = synthetic_images[idx, channel].detach().cpu().numpy()  # images
fig, axs = plt.subplots(nrows=1, ncols=3)
for ax in axs:
    ax.axis("off")
ax = axs[0]
ax.imshow(img[..., img.shape[2] // 2], cmap="gray")
ax = axs[1]
ax.imshow(img[:, img.shape[1] // 2, ...], cmap="gray")
ax = axs[2]
ax.imshow(img[img.shape[0] // 2, ...], cmap="gray")

In [ ]:
import torch
import numpy as np
from tqdm import tqdm

def generate_image(unet, autoencoder, scheduler, condition, num_inference_steps=50, guidance_scale=7.5):
    # Set models to evaluation mode
    unet.eval()
    autoencoder.eval()

    # Prepare the condition
    if condition.dim() == 1:
        condition = condition.unsqueeze(0)  # Add batch dimension
    if condition.dim() == 2:
        condition = condition.unsqueeze(1)  # Add sequence length dimension
    condition = condition.to(device)

    # Initialize latent with random noise
    latents = torch.randn(
        (1, unet.in_channels, 16, 16, 16),  # Adjust size based on your model
        device=device
    )

    # Set number of inference steps
    scheduler.set_timesteps(num_inference_steps)

    # Denoise the latents
    for t in tqdm(scheduler.timesteps):
        # Expand the latents for classifier free guidance
        latent_model_input = torch.cat([latents] * 2)
        latent_model_input = scheduler.scale_model_input(latent_model_input, t)

        # Predict the noise residual
        with torch.no_grad():
            noise_pred = unet(latent_model_input, t, encoder_hidden_states=condition.repeat(2, 1, 1))

        # Perform guidance
        noise_pred_uncond, noise_pred_cond = noise_pred.chunk(2)
        noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_cond - noise_pred_uncond)

        # Compute the previous noisy sample x_t -> x_t-1
        latents = scheduler.step(noise_pred, t, latents).prev_sample

    # Decode the latents to image
    with torch.no_grad():
        image = autoencoder.decode(latents / scale_factor)  # Use the scale_factor if you used it during training

    # Normalize the image
    image = (image / 2 + 0.5).clamp(0, 1)
    image = image.cpu().permute(0, 2, 3, 4, 1).squeeze().numpy()

    return image

# Example usage
condition = torch.tensor([0, 1, 0, 0, 0, 0, 0, 0,  # WAB type one-hot
                          1, 0,  # Sex one-hot
                          0, 1,  # Race one-hot
                          0.5, 60/100],  # Normalized WAB AQ and age
                         dtype=torch.float32)

generated_image = generate_image(unet, autoencoder, scheduler, condition)

# Visualize the generated image
import matplotlib.pyplot as plt

def plot_3d_image(image):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(image[:, :, image.shape[2]//2], cmap='gray')
    axes[0].set_title('Axial')
    axes[1].imshow(image[:, image.shape[1]//2, :], cmap='gray')
    axes[1].set_title('Coronal')
    axes[2].imshow(image[image.shape[0]//2, :, :], cmap='gray')
    axes[2].set_title('Sagittal')
    plt.show()

plot_3d_image(generated_image)

In [ ]:
import torch
import numpy as np
from tqdm import tqdm

def generate_image(unet, autoencoder, scheduler, condition, num_inference_steps=50, guidance_scale=7.5):
    # Set models to evaluation mode
    unet.eval()
    autoencoder.eval()

    # Prepare the condition
    if condition.dim() == 1:
        condition = condition.unsqueeze(0)  # Add batch dimension
    if condition.dim() == 2:
        condition = condition.unsqueeze(1)  # Add sequence length dimension
    condition = condition.to(device)

    # Initialize latent with random noise
    latents = torch.randn(
        (1, unet.in_channels, 16, 16, 16),  # Adjust size based on your model
        device=device
    )

    # Set number of inference steps
    scheduler.set_timesteps(num_inference_steps)

    # Denoise the latents
    for t in tqdm(scheduler.timesteps):
        # Expand the latents for classifier free guidance
        latent_model_input = torch.cat([latents] * 2)
        latent_model_input = scheduler.scale_model_input(latent_model_input, t)

        # Predict the noise residual
        with torch.no_grad():
            noise_pred = unet(latent_model_input, t, encoder_hidden_states=condition.repeat(2, 1, 1))

        # Perform guidance
        noise_pred_uncond, noise_pred_cond = noise_pred.chunk(2)
        noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_cond - noise_pred_uncond)

        # Compute the previous noisy sample x_t -> x_t-1
        latents = scheduler.step(noise_pred, t, latents).prev_sample

    # Decode the latents to image
    with torch.no_grad():
        image = autoencoder.decode(latents / scale_factor)  # Use the scale_factor if you used it during training

    # Normalize the image
    image = (image / 2 + 0.5).clamp(0, 1)
    image = image.cpu().permute(0, 2, 3, 4, 1).squeeze().numpy()

    return image

# Example usage
condition = torch.tensor([0, 1, 0, 0, 0, 0, 0, 0,  # WAB type one-hot
                          1, 0,  # Sex one-hot
                          0, 1,  # Race one-hot
                          0.5, 60/100],  # Normalized WAB AQ and age
                         dtype=torch.float32)

generated_image = generate_image(unet, autoencoder, scheduler, condition)

# Visualize the generated image
import matplotlib.pyplot as plt

def plot_3d_image(image):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(image[:, :, image.shape[2]//2], cmap='gray')
    axes[0].set_title('Axial')
    axes[1].imshow(image[:, image.shape[1]//2, :], cmap='gray')
    axes[1].set_title('Coronal')
    axes[2].imshow(image[image.shape[0]//2, :, :], cmap='gray')
    axes[2].set_title('Sagittal')
    plt.show()

plot_3d_image(generated_image)

## Clean-up data

In [ ]:
if directory is None:
    shutil.rmtree(root_dir)